# Services

Ein Service bietet eine stabile Adresse für wechselnde Pods. Lernziele: Selector, EndpointSlices und die drei Service-Typen unterscheiden.

![service-ports](service-ports.svg)

| Feld | Bedeutung |
|---|---|
| selector | Pod-Auswahl |
| port | Service-Port |
| targetPort | Port im Pod |
| nodePort | Port auf den Nodes |
| clusterIP | Virtuelle Service-IP |

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Deployment und ClusterIP

Zwei Replikate stellen Hostinformationen bereit. ClusterIP ist normalerweise nur clusterintern erreichbar. Port 9000 wird auf Port 80 abgebildet.

In [ ]:
%%bash
set -eo pipefail
cat deploy.yml cluster.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f deploy.yml
kubectl rollout status deployment/svc-test --timeout=180s
kubectl apply -f cluster.yml
kubectl get pods -l chapter=services -o wide
kubectl get svc web-clusterip

## 2. Zugriff und EndpointSlices

Ein Toolbox-Pod nutzt wget, da BusyBox kein curl enthält. EndpointSlices beschreiben die Backend-Adressen und deren Bereitschaft.

In [ ]:
%%bash
set -eo pipefail
kubectl run services-toolbox --image=busybox:1.36 --restart=Never --command -- sleep 3600
kubectl wait --for=condition=Ready pod/services-toolbox --timeout=180s

In [ ]:
%%bash
set -eo pipefail
kubectl exec services-toolbox -- wget -qO- http://web-clusterip:9000
kubectl get endpointslices -l kubernetes.io/service-name=web-clusterip -o wide

## 3. Negativbeispiel und Korrektur

Der Selector chapter=demo findet keine Pods. Der Service besitzt dennoch eine ClusterIP. Nach der Korrektur sollte der Zugriff funktionieren.

In [ ]:
%%bash
set -eo pipefail
cat cluster_demo.yml
kubectl apply -f cluster_demo.yml
kubectl get endpointslices -l kubernetes.io/service-name=web-clusterip-demo -o yaml
kubectl exec services-toolbox -- wget -T 3 -qO- http://web-clusterip-demo:9001 || echo 'Kein erreichbares Backend'

In [ ]:
%%bash
set -eo pipefail
kubectl patch svc web-clusterip-demo -p '{"spec":{"selector":{"chapter":"services"}}}'
for i in $(seq 1 15); do if kubectl exec services-toolbox -- wget -T 3 -qO- http://web-clusterip-demo:9001; then exit 0; fi; sleep 2; done
exit 1

## 4. NodePort

Der NodePort 30090 ist auf erreichbaren Nodes verfügbar. Der Service-Port bleibt 9000. Neue TCP-Verbindungen können verschiedene Pods erreichen.

In [ ]:
%%bash
set -eo pipefail
cat nodeport.yml
kubectl apply -f nodeport.yml
kubectl get nodes -o wide
kubectl get svc web-nodeport

In [ ]:
%%bash
set -eo pipefail
NODE_IP=$(kubectl get nodes -o jsonpath='{.items[0].status.addresses[?(@.type=="InternalIP")].address}')
for i in $(seq 1 10); do curl -fsS --max-time 5 "http://$NODE_IP:30090/" | grep Hostname; done | sort | uniq -c

## 5. LoadBalancer

K3s ServiceLB stellt Port 9000 auf verfügbaren Nodes bereit. Bei Pending ServiceLB und Portkonflikte prüfen. Der Host muss die angezeigte IP erreichen können.

In [ ]:
%%bash
set -eo pipefail
cat lb.yml
kubectl apply -f lb.yml
kubectl wait --for=jsonpath='{.status.loadBalancer.ingress[0].ip}' service/svc-lb --timeout=120s
kubectl get svc svc-lb

In [ ]:
%%bash
set -eo pipefail
IP=$(kubectl get svc svc-lb -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
for i in $(seq 1 10); do curl -fsS --max-time 5 "http://$IP:9000/" | grep Hostname; done | sort | uniq -c

## 6. Aufräumen

Alle drei Zugriffswege führen zu denselben Backends. Die Anzahl der Antworten pro Pod ist statistisch.

In [ ]:
%%bash
set -eo pipefail
kubectl get svc web-clusterip web-nodeport svc-lb
kubectl delete -f lb.yml -f nodeport.yml -f cluster_demo.yml -f cluster.yml -f deploy.yml
kubectl delete pod services-toolbox
kubectl get pods,svc -l chapter=services

## Fazit

Services entkoppeln Clients von Pod-Adressen. ClusterIP, NodePort und LoadBalancer unterscheiden sich im Zugriffsweg, nicht im grundlegenden Selector-Prinzip.